# Lab 7: Report

## Cold Working & Annealing

This is a guided **teaching notebook** with a mix of written response and coding. It follows the Lab 7 handout and is designed to help you process the data, make the required figure, and answer the lab-report questions.


## How to use this notebook

This notebook contains two main cell types:

- **Markdown cells** contain instructions, equations, figures, and your written answers.
- **Code cells** contain Python. Run a selected cell with **Shift+Enter**.

You will see four kinds of prompts:

- **`YOUR RESPONSE`**: something you must complete. Double-click the Markdown cell and replace the prompt with your answer. The three graded items are also marked with their point values.
- **`STUDENT`**: a comment in a code cell marking a line you must complete. Lines containing `YOUR RESPONSE` will give a `SyntaxError` until you replace them.
- **`ENGINEERING CHECK`**: stop and inspect the result before continuing. Is it physically reasonable?
- **`CHECKPOINT`** *(new)*: a code cell that tests the code you just wrote on a simple example with a known answer. If it prints `CHECKPOINT PASSED`, continue. If it raises an `AssertionError`, fix your code before moving on.

AI may not be used to analyze the experimental data or answer the written questions. If you run into a coding error, you may use AI to troubleshoot the code (provide the code and the error message).

## Lab objectives

From the Lab 7 handout:

- Study the plastic deformation of ductile metals and the effect of this deformation on their mechanical properties.
- Cold roll copper and brass (70 wt% Cu / 30 wt% Zn) to various thicknesses.
- Measure strain and %CW (percent cold work) and determine how the degree of cold working affects hardness.
- Observe the change in hardness that results from annealing.
- Relate the changes in mechanical properties to changes in the microstructure.


Reading: Callister, *Materials Science and Engineering: An Introduction*, Sections 7.10–7.13.

## 1. Load the data into the notebook

The data are supplied in the Excel workbook `Lab7_Data.xlsx`. Do not export the data to `.csv`.

1. Open the workbook in Excel and inspect it. Each specimen occupies a block of rows (thickness, width, area, %CW, three HRB readings, average HRB), and each column is one **step** of the experiment (pass 0 through 8, plus the annealing steps).
1. Place the Excel file in the same folder as this notebook.
1. Update `DATA_FILE` below if your file name is different.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
DATA_FILE = "Lab7_Data.xlsx"    # STUDENT: update the text string to match your Excel workbook

data_path = Path.cwd() / DATA_FILE

if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. "
        "Place the Excel workbook in the same folder as this notebook and confirm the file name matches the code."
    )

if data_path.suffix.lower() != ".xlsx":
    raise ValueError("Invalid file type: please provide an .xlsx workbook.")

print(f"Found data file: {data_path.name}")

### 2.1 Reorganize the workbook into a single table

The workbook is laid out for people (one wide block per specimen). For analysis it is easier to have a **long** table: one row per specimen per step, one column per quantity.

Run the cell below without changing it. You are **not expected to write this code**, but read the table below so you know what each column of the new table `lab7` means:

| imported data names | meaning |
|:---|:---|
| `material` | copper or brass |
| `anneal` | anneal condition of the specimen: `RT`, `400 °C`, or `500 °C` |
| `specimen` | material + anneal condition, e.g. `copper 400 °C` |
| `step` | order of the measurement (0, 1, 2, …) |
| `step_label` | the column heading in the workbook (e.g. `5 (post-an)`) |
| `stage` | `rolled` (measured after a rolling pass) or `annealed` (measured right after a heat treatment) |
| `thickness_in`, `width_in` | caliper measurements in inches |
| `HRB1`, `HRB2`, `HRB3` | the three Rockwell B readings (blank if not measurable) |
| `too_soft` | `True` if the workbook says the specimen was too soft to measure on the B scale |

In [ ]:
def load_lab7_workbook(path):
    raw = pd.read_excel(path, header=None)

    # Merged cells only store a value in the top-left cell: fill material names downward.
    material_column = raw.iloc[:, 0].ffill()
    step_labels = raw.iloc[1]               # row 2 in Excel holds the step headings
    step_columns = range(3, raw.shape[1])   # columns D, E, F, ... hold the steps

    # Each specimen block starts on the row whose label (column C) is "thickness (in)".
    block_starts = [r for r in range(raw.shape[0])
                    if str(raw.iloc[r, 2]).strip().lower().startswith("thickness")]

    records = []
    for start in block_starts:
        material = str(material_column.iloc[start]).strip().lower()
        anneal = raw.iloc[start, 1]
        anneal = "RT" if str(anneal).strip().upper() == "RT" else f"{int(anneal)} °C"

        for step, col in enumerate(step_columns):
            label = step_labels.iloc[col]
            if isinstance(label, float) and label.is_integer():
                label = int(label)            # show pass numbers as 1, 2, ... rather than 1.0, 2.0
            label = str(label).strip()
            if label == "nan":
                continue
            hrb_raw = [raw.iloc[start + k, col] for k in (4, 5, 6)]   # HRB1-3 rows
            thickness = pd.to_numeric(raw.iloc[start, col], errors="coerce")
            width = pd.to_numeric(raw.iloc[start + 1, col], errors="coerce")
            hrb = [pd.to_numeric(v, errors="coerce") for v in hrb_raw]
            too_soft = any("soft" in str(v).lower() for v in hrb_raw)

            # skip steps that were never measured for this specimen
            if np.isnan(thickness) and np.all(np.isnan(hrb)) and not too_soft:
                continue

            records.append({
                "material": material,
                "anneal": anneal,
                "specimen": f"{material} {anneal}",
                "step": step,
                "step_label": label,
                "stage": "annealed" if ("post" in label or "anneal" in label) else "rolled",
                "thickness_in": thickness,
                "width_in": width,
                "HRB1": hrb[0], "HRB2": hrb[1], "HRB3": hrb[2],
                "too_soft": too_soft,
            })

    return pd.DataFrame(records)


lab7 = load_lab7_workbook(data_path)
SPECIMENS = list(lab7["specimen"].unique())

print(f"Loaded {len(lab7)} measurements for {len(SPECIMENS)} specimens:")
for s in SPECIMENS:
    print("  ", s)



In [ ]:
# run this cell to display the data for the any specimen in the list. change the index in SPECIMENS[...] to view other specimens.

display(lab7[lab7["specimen"] == SPECIMENS[0]])

**`ENGINEERING CHECK`**

Compare the table above with the same specimen in Excel before continuing. Notice that:

- the `5 (post-an)` and `last anneal.` rows have no thickness or width, because annealing does not change the dimensions;
- the `RT` specimens have no `5 (post-an)` row, because they were not heated;
- some readings are marked *"too soft / graph 0"*: after some anneals the specimen was too soft to give a valid reading on the Rockwell B scale.

### 2.2 Complete the dimensions for the annealed steps

Because annealing does not change the dimensions, an annealed step has the same thickness and width as the rolled step just before it.

Pandas can copy the last valid value forward in a column with the method `.ffill()` ("forward fill"). Because the table contains six specimens, the fill must be done **within each specimen**, which is what `lab7.groupby("specimen")[...]` does.

Complete the two lines below.

In [ ]:
# STUDENT: forward-fill the thickness and width within each specimen
lab7["thickness_in"] = lab7.groupby("specimen")["thickness_in"].YOUR RESPONSE      # STUDENT: complete with the forward-fill method
lab7["width_in"] = lab7.groupby("specimen")["width_in"].YOUR RESPONSE              # STUDENT: complete with the forward-fill method

missing = lab7[["thickness_in", "width_in"]].isna().sum().sum()
print("Missing dimensions remaining:", missing)
display(lab7[lab7["stage"] == "annealed"][["specimen", "step_label", "thickness_in", "width_in"]])

## 3. Calculate %CW and average hardness

### 3.1 Percent cold work

The handout specifies that **%CW is based on the change in cross-sectional area relative to the initial (not cold-rolled) cross-section**:

$$\%\text{CW} = \frac{A_0 - A_d}{A_0}\times 100$$

where $A_0$ is the original cross-sectional area and $A_d$ is the cross-sectional area after deformation. For the rectangular strips, $A = t \times w$ (thickness × width).

Complete the code below:

1. Calculate the cross-sectional area in in² (the workbook's `A (in2)` row).
1. Write a function `percent_cold_work(area, area_0)` that returns %CW using the equation above.

**Variables for your code**:

- `lab7["thickness_in"]`, `lab7["width_in"]`: dimensions in inches
- `area`: current cross-sectional area, $A_d$
- `area_0`: initial (pass 0) cross-sectional area of the same specimen, $A_0$

In [ ]:
lab7["area_in2"] = YOUR RESPONSE     # STUDENT: cross-sectional area in in^2


def percent_cold_work(area, area_0):
    # STUDENT: return the percent cold work (in percent, not as a fraction)
    return YOUR RESPONSE

In [ ]:
# CHECKPOINT: run without changing. These examples have known answers.
assert np.isclose(percent_cold_work(50.0, 100.0), 50.0), "A specimen reduced to half its area has 50 %CW."
assert np.isclose(percent_cold_work(100.0, 100.0), 0.0), "An unrolled specimen has 0 %CW."
assert np.isclose(percent_cold_work(75.0, 100.0), 25.0), "Check that your function returns percent, not a fraction."
print("CHECKPOINT PASSED")

Run the cell below to apply your function to every specimen. The initial area $A_0$ is taken from the first step (pass 0) of each specimen.

In [ ]:
lab7["area0_in2"] = lab7.groupby("specimen")["area_in2"].transform("first")
lab7["%CW"] = percent_cold_work(lab7["area_in2"], lab7["area0_in2"])

display(lab7[["specimen", "step_label", "stage", "thickness_in", "width_in", "area_in2", "%CW"]].round(4))

**`ENGINEERING CHECK`**

- %CW should start at 0 for every specimen and increase with every rolling pass.
- Annealed steps should have the same %CW as the rolled step just before them.
- The handout asks for rolling until just over 50 %CW. Check the final %CW of each specimen.

### 3.2 Average hardness

The handout asks for **at least three measurements per specimen** and an **average hardness value**. Compute the mean and the standard deviation of the three readings for each step.

Because the three readings are in three **columns** of the same row, you must tell pandas to work across the columns with `axis=1`, for example `lab7[HRB_COLUMNS].mean(axis=1)`.

**Functions**:

- `.mean(axis=1)`: average across columns
- `.std(axis=1)`: sample standard deviation across columns
- `.count(axis=1)`: number of non-missing values across columns

Readings that were too soft to measure are **graphed as 0**, as instructed in the data workbook. The plot in Section 4 marks these points with an open (hollow) marker so a reader can see that they are not real measurements.

In [ ]:
TOO_SOFT_HRB = 0          # value used for readings that were too soft to measure ("graph 0" in the workbook)

HRB_COLUMNS = ["HRB1", "HRB2", "HRB3"]

lab7["HRB_mean"] = YOUR RESPONSE          # STUDENT: mean of the three readings
lab7["HRB_std"] = YOUR RESPONSE            # STUDENT: sample standard deviation of the three readings
lab7["HRB_n"] = YOUR RESPONSE            # STUDENT: number of valid readings

# readings that were too soft to measure are graphed as 0 (no standard deviation)
lab7.loc[lab7["too_soft"], "HRB_mean"] = TOO_SOFT_HRB
lab7.loc[lab7["too_soft"], "HRB_std"] = np.nan

display(lab7[["specimen", "step_label", "%CW", "HRB_mean", "HRB_std", "HRB_n", "too_soft"]].round(2))

In [ ]:
# CHECKPOINT: run without changing.
test = pd.DataFrame({"HRB1": [40.0], "HRB2": [42.0], "HRB3": [44.0]})
assert np.isclose(test[HRB_COLUMNS].mean(axis=1)[0], 42.0)
first = lab7.iloc[0]
assert np.isclose(first["HRB_mean"], first[HRB_COLUMNS].astype(float).mean()), \
    "HRB_mean should be the average of HRB1, HRB2 and HRB3 in the same row (did you use axis=1?)."
assert lab7["HRB_n"].max() == 3, "HRB_n should count the readings in each row (did you use axis=1?)."
print("CHECKPOINT PASSED")

## 4. Rockwell B hardness versus %CW for all six specimens

This is the first graded item in the handout. Your figure **is** your answer: it must show all six specimens (three copper and three brass), with labeled axes and units, a legend that identifies each specimen, and a caption.

Plot the **average HRB** against **%CW**, with vertical error bars equal to the standard deviation of the three readings. Because an annealed step has the same %CW as the rolled step just before it, an anneal appears on the plot as a **vertical drop** in hardness at constant %CW.

First, set your plotting choices. Give each of the six specimens its own color and marker.

In [ ]:
# STUDENT: choose a color, marker, and line style for each of the six specimens
COLOR = {
    "copper RT": "k", "copper 400 °C": "tab:red", "copper 500 °C": "tab:blue",
    "brass RT": "tab:cyan", "brass 400 °C": "tab:pink", "brass 500 °C": "tab:olive",
}
MARKER = {
    "copper RT": "s", "copper 400 °C": "o", "copper 500 °C": "^",
    "brass RT": "v", "brass 400 °C": "<", "brass 500 °C": ">",
}
LINE_STYLE = {
    "copper RT": "-", "copper 400 °C": "-", "copper 500 °C": "-",
    "brass RT": "--", "brass 400 °C": "--", "brass 500 °C": "--",
}

TICK_LABEL_FONT_SIZE = 12
AXIS_LABEL_FONT_SIZE = 14
TITLE_LABEL_FONT_SIZE = 16

plt.rcParams.update({"axes.titlesize": TITLE_LABEL_FONT_SIZE,
                     "axes.labelsize": AXIS_LABEL_FONT_SIZE,
                     "xtick.labelsize": TICK_LABEL_FONT_SIZE,
                     "ytick.labelsize": TICK_LABEL_FONT_SIZE,
                     "legend.fontsize": TICK_LABEL_FONT_SIZE - 1,
                    })
print("Plot settings updated.")

### Rockwell B hardness vs. %CW (20 points)

**`YOUR RESPONSE`**: Plot the Rockwell B hardness versus %CW for all six specimens (three copper and three brass).

Complete the plotting code below.

**Variables available inside the loop** (for one specimen at a time):

- `data["%CW"]`: x values
- `data["HRB_mean"]`: y values
- `data["HRB_std"]`: vertical error bars

**Functions**:

- `ax.errorbar(x, y, yerr=..., ...)`: plot data with error bars (the formatting arguments are already given)
- `ax.set_xlabel()`, `ax.set_ylabel()`, `ax.set_title()`: axis labels and title

Refer to the plotting tips in the Lab 2 notebook. Finish by writing a caption that tells the reader what is plotted, what the vertical drops are, what the error bars represent, and what the hollow markers mean.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))

for specimen in SPECIMENS:
    data = lab7[lab7["specimen"] == specimen].sort_values("step")

    # STUDENT: supply the x values, y values, and the vertical error bars
    ax.errorbar(
        YOUR RESPONSE,                      # STUDENT: x values
        YOUR RESPONSE,                 # STUDENT: y values
        yerr=YOUR RESPONSE,             # STUDENT: vertical error bars
        color=COLOR[specimen],
        marker=MARKER[specimen],
        linestyle=LINE_STYLE[specimen],
        capsize=3,
        label=specimen,
    )

    # mark readings that were too soft to measure with a hollow marker
    soft = data[data["too_soft"]]
    if not soft.empty:
        ax.scatter(soft["%CW"], soft["HRB_mean"], s=110, marker=MARKER[specimen],
                   facecolors="white", edgecolors=COLOR[specimen], zorder=5, label="_")

# one legend entry explaining the hollow markers
ax.scatter([], [], s=110, marker="o", facecolors="white", edgecolors="gray",
           label=f"too soft for HRB (graphed as {TOO_SOFT_HRB})")

ax.set_title("YOUR RESPONSE")     # STUDENT: descriptive title
ax.set_xlabel("YOUR RESPONSE")                 # STUDENT: x-axis label with units
ax.set_ylabel("YOUR RESPONSE")       # STUDENT: y-axis label with units
ax.set_xlim(-2, lab7["%CW"].max() + 3)
ax.set_ylim(bottom=-3)
ax.grid(alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

print("Figure 1. STUDENT: GIVE A SUITABLE CAPTION.")   # STUDENT: provide a caption

**`ENGINEERING CHECK`**

Inspect your figure before continuing:

1. Before the first anneal, all three specimens of the same material were processed the same way. Do their curves overlap?
1. Where does the hardness drop on annealing, and by how much? Which anneals produced only a small drop, and which made the specimen too soft to measure?
1. Can every curve and every point be distinguished? If not, adjust your colors, markers, or line styles above.

## 5. Microstructure

Micrographs of copper and brass before cold rolling, at ≈25 %CW, and at ≈50 %CW (all without annealing) are provided on D2L.

1. Download the six micrographs and place them in the same folder as this notebook.
1. Update the file names in `MICROGRAPHS` to match your files.
1. Run the cell to display them in one figure, and give the figure a caption.

The rolling direction is horizontal in these images.

In [ ]:
# STUDENT: update the file names to match the micrographs you downloaded
MICROGRAPHS = {
    ("copper", "0 %CW"): "Cu_0CW.jpg",
    ("copper", "25 %CW"): "Cu_25CW.jpg",
    ("copper", "50 %CW"): "Cu_50CW.jpg",
    ("brass", "0 %CW"): "Brass_0CW.jpg",
    ("brass", "25 %CW"): "Brass_25CW.jpg",
    ("brass", "50 %CW"): "Brass_50CW.jpg",
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, ((material, cw), filename) in zip(axes.ravel(), MICROGRAPHS.items()):
    path = Path.cwd() / filename
    if path.exists():
        ax.imshow(plt.imread(path))
    else:
        ax.text(0.5, 0.5, f"{filename}\nnot found", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(f"{material.capitalize()}, {cw}")
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Figure 2. STUDENT: GIVE A SUITABLE CAPTION.")   # STUDENT: provide a caption

### Microstructural change with increasing %CW (10 points)

**`YOUR RESPONSE`**: Describe the microstructural change with increasing %CW by comparing the micrographs taken in the course of the experiment.

*Replace this text with your response in paragraph form.*

## 6. Effect of composition, cold work, and annealing on hardness

The last graded item asks you to use your previous analyses to explain the effects of composition, extent of cold work, and annealing treatment on the measured hardness. The two tables in this section pull the relevant numbers out of your data so you can use them as evidence. They are **not** graded separately.

### 6.1 Hardness change on annealing

For every anneal, compare the hardness of the as-received specimen ($\text{HRB}_0$), the hardness just before the anneal ($\text{HRB}_\text{pre}$), and just after it ($\text{HRB}_\text{post}$).

Complete the two calculations below:

- `work_hardening`: the increase in hardness from cold work before the anneal, $\text{HRB}_\text{pre} - \text{HRB}_0$
- `delta_hrb`: the change in hardness on annealing, $\text{HRB}_\text{post} - \text{HRB}_\text{pre}$

In [ ]:
records = []
for specimen in SPECIMENS:
    data = lab7[lab7["specimen"] == specimen].sort_values("step").reset_index(drop=True)
    hrb_0 = data.loc[0, "HRB_mean"]

    for i in data.index[data["stage"] == "annealed"]:
        hrb_pre = data.loc[i - 1, "HRB_mean"]
        hrb_post = data.loc[i, "HRB_mean"]

        work_hardening = YOUR RESPONSE          # STUDENT: hardness gained by cold work before the anneal
        delta_hrb = YOUR RESPONSE            # STUDENT: change in hardness on annealing (post minus pre)

        records.append({
            "specimen": specimen,
            "treatment": "500 °C, 3 min" if "last" in data.loc[i, "step_label"] else f"{data.loc[i, 'anneal']}, 5 min",
            "%CW at anneal": data.loc[i, "%CW"],
            "HRB_0": hrb_0,
            "HRB_pre": hrb_pre,
            "HRB_pre - HRB_0": work_hardening,
            "HRB_post": hrb_post,
            "ΔHRB on annealing": delta_hrb,
            "too soft after anneal?": data.loc[i, "too_soft"],
        })

anneal_table = pd.DataFrame(records).set_index("specimen")
display(anneal_table.round(1))

**`ENGINEERING CHECK`**: For rows marked `too soft after anneal? = True`, `HRB_post` is the graphed value of 0, not a real measurement. What you can say for certain is that the hardness fell below the lower limit of the Rockwell B scale.

### 6.2 Annealing temperature compared with the recrystallization temperature

Whether an anneal produces only **recovery** or full **recrystallization** depends on the annealing temperature (and time) relative to the recrystallization temperature $T_R$ of the material. Recrystallization usually occurs between about 0.3 and 0.7 of the absolute melting temperature $T_m$.

1. Look up $T_R$ and $T_m$ for copper and for brass in Callister, Table 7.2. Note the purity or composition of each material in the table, and how it compares with the materials used in this lab.
1. Write a function to convert °C to K, and calculate the homologous temperature $T/T_m$ (both in K) for each anneal.

In [ ]:
# STUDENT: enter the values from Callister Table 7.2 in °C
MELTING_T_C = {
    "copper": YOUR RESPONSE,          # STUDENT: melting temperature of copper
    "brass": YOUR RESPONSE,            # STUDENT: melting temperature of brass
}
RECRYSTALLIZATION_T_C = {
    "copper": YOUR RESPONSE,           # STUDENT: recrystallization temperature of copper
    "brass": YOUR RESPONSE,            # STUDENT: recrystallization temperature of brass
}
ANNEAL_T_C = {"RT": 22, "400 °C": 400, "500 °C": 500}


def celsius_to_kelvin(t_c):
    # STUDENT: convert a temperature from °C to K
    return YOUR RESPONSE


records = []
for material in ["copper", "brass"]:
    for anneal, t_c in ANNEAL_T_C.items():
        records.append({
            "material": material,
            "anneal": anneal,
            "T / K": celsius_to_kelvin(t_c),
            "T_m / K": celsius_to_kelvin(MELTING_T_C[material]),
            "T / T_m": YOUR RESPONSE,     # STUDENT: homologous temperature
            "T_R / °C": RECRYSTALLIZATION_T_C[material],
            "above T_R?": t_c > RECRYSTALLIZATION_T_C[material],
        })

temperature_table = pd.DataFrame(records).set_index(["material", "anneal"])
display(temperature_table.round(2))

In [ ]:
# CHECKPOINT: run without changing.
assert np.isclose(celsius_to_kelvin(0), 273.15), "0 °C is 273.15 K."
assert np.isclose(celsius_to_kelvin(-273.15), 0), "Absolute zero is -273.15 °C."
print("CHECKPOINT PASSED")

### Effect of composition, extent of cold work, and annealing treatment (30 points)

**`YOUR RESPONSE`**: Using the previous analyses, explain the effect of composition, extent of cold work, and annealing treatment (temperature and time) on the measured hardness.

Address all three factors, and support your explanation with your data (Figure 1, Figure 2, and the tables in Section 6). You may organize your answer under the three headings below.

**Composition**

*Replace this text with your response in paragraph form.*

**Extent of cold work**

*Replace this text with your response in paragraph form.*

**Annealing treatment (temperature and time)**

*Replace this text with your response in paragraph form.*

## CONTRIBUTION STATEMENT

*List each contributor and what they did.*

**`YOUR RESPONSE`**

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. Make sure all cells have been run (**Kernel → Restart & Run All**), all `CHECKPOINT` cells print `CHECKPOINT PASSED`, and all figures are displayed properly.